In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "workspace.default.pinterest_posts_bronze"
SILVER_TABLE = "workspace.default.pinterest_post_tags_silver"

posts = spark.table(BRONZE_TABLE)

# Optional: only if these columns exist in bronze
if "_corrupt_record" in posts.columns:
    posts = posts.where(F.col("_corrupt_record").isNull())

posts = (
    posts
    .where(F.col("date_posted").isNotNull())
    .withColumn("hashtags", F.trim(F.col("hashtags")))
    .where(F.col("hashtags").isNotNull())
    .where((F.col("hashtags") != "") & (F.lower(F.col("hashtags")) != F.lit("null")))
)

tags = (
    posts
    .withColumn("tag", F.explode(F.split(F.col("hashtags"), ",")))
    .withColumn("tag", F.trim(F.col("tag")))
    .where(F.length("tag") > 1)
    .where(F.lower(F.col("tag")) != F.lit("null"))
    .select(
        "user_id", "user_name", "user_url",
        "date_posted",
        "post_year", "post_quarter",
        "tag",
        "source_file_path",
        "_ingest_ts"
    )
)

tags.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

print("silver rows:", tags.count())
print("distinct tags:", tags.select("tag").distinct().count())